# Reproducing Timme et al. (2016) from CRCNS ssc-3

**Paper.** Timme NM, Ito S, Myroshnychenko M, Nigam S, Shimono M, Yeh F-C, Hottowy P, Litke AM,
Beggs JM (2016). High-degree neurons feed cortical computations. *PLoS Comput Biol* 12(5): e1004858.
[doi:10.1371/journal.pcbi.1004858](https://doi.org/10.1371/journal.pcbi.1004858). The claims file
was written from the version of record (PMC4861348, Europe PMC full text). Its displayed
equations, missing from that flattened text, were read from the version of record's JATS XML.
The TE procedure the paper reuses is that of its reference [44], Timme et al. (2014), PLoS One 9:
e115764 (PMC4275261).

**Data set.** [CRCNS ssc-3](https://doi.org/10.6080/K07D2S2F), `ssc-3/live/data.tar.gz` in the public
CRCNS bucket, streamed and decompressed in memory. CRCNS data sets are not versioned, so the version
is recorded as `unversioned`; the paper cites the data set by its DOI and no version, and this is
the only copy the bucket holds (`ssc-3/live/`, no `old/`). The MD5 sum of the archive, from the data
set's `checksums.md5`, is `1a5bd3d304c5442290a7c9c1f1b5def6`; the loading cell checks the streamed
bytes against it. The archive holds 25 MATLAB files, one per recording.

**Claims.**

| Claim | Tested here |
|---|---|
| data-recordings | tested |
| fig4ac-te-lognormal | tested |
| fig4bd-heavy-tailed-degree | tested |
| fig4bd-out-tail-longer | tested |
| fig5a-in-degree-uncorrelated | tested |
| fig5b-out-degree-positive | tested |
| fig5ab-correlations-beyond-null | tested |
| fig5cd-pooled-trends | tested |
| fig6-information-gain-not-rising | tested, with sampled receivers and combinations |
| table1-pid-examples | tested (no recorded data) |
| fig7b-connectivity-patterns | tested (model simulation) |
| fig7c-degree-modified-rule-matches | tested (model simulation) |
| fig7d-degree-rule-highest-conditioned-mi | tested (model simulation) |
| figs1-larger-model | tested (model simulation) |

How the TE networks are built (bins, past states, jitter, sub-networks) and where each choice comes
from is in the docstring of `hdn.py`. The significance test uses the paper's 5000 jittered
surrogates per pair, with one approximation needed to fit the time limit: for pairs whose jitter
moves more than 50 relevant sender spikes, the jittered counts are drawn from a normal distribution
with their exact moments, in place of jittering every spike. The first cell of the TE networks section compares the
two on two recordings.

In [ ]:
import itertools
import math
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import curve_fit

import hdn

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 200)
started = time.time()
recs, md5 = hdn.load_all()
print(f"CRCNS {hdn.DATASET} ({hdn.DATASET_DOI}), version {hdn.DATASET_VERSION}: {len(recs)} recordings "
      f"streamed in {time.time() - started:.0f} s")
print("MD5 of the streamed archive:", md5, "| checksums.md5:", hdn.DATA_MD5, "| equal:", md5 == hdn.DATA_MD5)

## Data check

In [ ]:
rows = []
for r in recs:
    n = np.array([len(s) for s in r["spikes"]])
    allsp = np.concatenate(r["spikes"])
    rows.append({
        "file": r["name"], "recording_id": r["recording_id"], "nNeurons": r["n_neurons_field"],
        "trains": len(r["spikes"]), "length_ms": r["length_ms"], "first_ms": allsp.min(), "last_ms": allsp.max(),
        "spikes_min": n.min(), "spikes_max": n.max(), "under_100": int((n < 100).sum()),
        "sorted": all(np.all(np.diff(s) >= 0) for s in r["spikes"]),
        "on_50us_grid": bool(np.allclose(np.round(allsp * 20) / 20, allsp)),
        "x_um": f"{np.nanmin(r['x']):.0f} to {np.nanmax(r['x']):.0f}", "y_um": f"{np.nanmin(r['y']):.0f} to {np.nanmax(r['y']):.0f}",
        "timescale": r["timescale"], "sampling": r["sampling"]})
info = pd.DataFrame(rows)
print(info.to_string(index=False))
print("\nnNeurons equals the number of spike trains in every file:", bool((info.nNeurons == info.trains).all()))
print("Spike times sorted within every train:", bool(info.sorted.all()), "| on the 0.05 ms grid (20 kHz):", bool(info.on_50us_grid.all()))
print("Every spike within [0, recordinglength]:", bool(((info.first_ms >= 0) & (info.last_ms <= info.length_ms)).all()))
print("Recording lengths (ms):", info.length_ms.value_counts().to_dict())
print("Recording dates:", sorted(set(r[:10] for r in info.recording_id)))
# recordings deposited twice: identical spike trains in two files
sig = {}
for r in recs:
    for k, s in enumerate(r["spikes"]):
        sig.setdefault((len(s), round(float(s[0]), 2), round(float(s[-1]), 2)), []).append((r["name"], k))
shared = [v for v in sig.values() if len({f for f, _ in v}) > 1]
print("Spike trains with the same count, first and last spike in two files:", len(shared))
print("Distinct recording IDs:", info.recording_id.nunique(), "of", len(info))

The descriptor says the data are in ms at 20 kHz, with recordings of "approximately 1 hour"
(3600000 ms "for most data sets") and positions in um with the origin at the array's centre on a
2 mm by 1 mm array. The values agree: spike times sit on the 0.05 ms grid, positions span about
-1100 to 1100 um in x and -600 to 600 um in y, 24 recordings last 3600000 ms and one (DataSet19)
2887498 ms (48 min). The first ten recordings have no spike in their first 8 s. No spike train
appears in two files and the 25 recording IDs are distinct, so no recording is deposited twice.
The files hold spike times only: there are no trials or other data streams whose clock could
differ from the spikes', so the clock check of the protocol does not apply.

## data-recordings

> The analysis used 25 recordings of somatosensory cortex cultures, with 7735 neurons in total after
> removing neurons with fewer than 100 spikes in the 60 min recording, an average of 309.4 neurons
> per recording and an average firing rate of 2.10 Hz.

In [ ]:
counts = np.array([len(r["spikes"]) for r in recs])
spk = [np.array([len(s) for s in r["spikes"]]) for r in recs]
keep = [n >= 100 for n in spk]
rate_len = np.concatenate([n / (r["length_ms"] / 1000) for n, r in zip(spk, recs)])
rate_60 = np.concatenate([n / 3600.0 for n in spk])
print("Recordings:", len(recs))
print("Neurons as deposited:", counts.sum(), "| mean per recording:", round(counts.mean(), 1))
print("Neurons with 100 or more spikes:", sum(k.sum() for k in keep), "(the deposit already applies the threshold)")
print(f"Mean rate per neuron, (a) spikes over each recording's length: {rate_len.mean():.3f} Hz; "
      f"(b) spikes over 60 min: {rate_60.mean():.3f} Hz")
print(f"Rates range from {rate_len.min():.3f} to {rate_len.max():.1f} Hz")

**Result: reproduced.** The data set holds 25 recordings and 7735 neurons (309.4 per recording), every one with at least 100 spikes, so the deposit already applies the paper's threshold. The mean rate is 2.103 Hz with spikes counted over 60 min (reading b) and 2.132 Hz over each recording's own length (reading a); both are within 0.21 Hz of 2.10 Hz.

## TE networks

TE from every neuron to every other, at both time scales, with the jitter test (`hdn.te_network`).
The first cell checks the normal approximation against the full jitter of every spike on two
recordings at the 1.6 ms bins: the decision of the test is compared pair by pair.

In [ ]:
check = []
for name in ["DataSet1", "DataSet7"]:
    r = next(x for x in recs if x["name"] == name)
    trains, T = hdn.bin_trains(r["spikes"], 1.6, r["length_ms"])
    t = time.time()
    exact = hdn.significant(hdn.te_network(trains, T, hdn.N_SURROGATES, hdn.MAX_EXCEED, 1, 10**9)[3])
    t_exact = time.time() - t
    t = time.time()
    approx = hdn.significant(hdn.te_network(trains, T, hdn.N_SURROGATES, hdn.MAX_EXCEED, 1, 50)[3])
    t_approx = time.time() - t
    check.append({"recording": name, "pairs": exact.size - len(exact), "significant_exact": int(exact.sum()),
                  "significant_approx": int(approx.sum()), "both": int((exact & approx).sum()),
                  "exact_only": int((exact & ~approx).sum()), "approx_only": int((~exact & approx).sum()),
                  "seconds_exact": round(t_exact), "seconds_approx": round(t_approx)})
print(pd.DataFrame(check).to_string(index=False))

In [ ]:
nets = {}
rows = []
t_all = time.time()
for scale, bin_ms in hdn.TIME_SCALES.items():
    for i, r in enumerate(recs):
        trains, T = hdn.bin_trains(r["spikes"], bin_ms, r["length_ms"])
        t = time.time()
        te, nte, ran, exceed = hdn.te_network(trains, T, hdn.N_SURROGATES, hdn.MAX_EXCEED, 1, 50)
        sig = hdn.significant(exceed)
        nets[scale, r["name"]] = {"trains": trains, "T": T, "te": te, "nte": nte, "sig": sig}
        n = len(trains)
        rows.append({"scale": scale, "recording": r["name"], "neurons": n, "connections": int(sig.sum()),
                     "density": sig.sum() / (n * (n - 1)), "expected_by_chance": round(0.001 * n * (n - 1)),
                     "mean_total_degree": 2 * sig.sum() / n, "seconds": round(time.time() - t)})
net_info = pd.DataFrame(rows)
print(net_info.round(3).to_string(index=False))
print(f"\n{time.time() - t_all:.0f} s")
print(net_info.groupby("scale")[["connections", "density", "mean_total_degree"]].describe().round(3).T.to_string())

In [ ]:
res = {}
t_all = time.time()
for (scale, name), net in nets.items():
    seed = 1000 * list(hdn.TIME_SCALES).index(scale) + int(name.replace("DataSet", ""))
    res[scale, name] = hdn.analyse_recording(net["trains"], net["T"], net["sig"], net["nte"], seed)
print(f"sub-networks, synergy and information gains in {time.time() - t_all:.0f} s")
summ = pd.DataFrame([{"scale": s, "recording": n, "subnetworks": len(a["subs"]), "restarts": a["restarts"],
                      "subnet_triples": len(a["sub_triples"]), "full_triples_total": a["n_full_total"],
                      "full_triples_used": len(a["full_triples"])} for (s, n), a in res.items()])
print(summ.to_string(index=False))

## fig4ac-te-lognormal

> The significant TE connection weights, raw and normalized, are roughly log-normally distributed at
> both time scales and vary over several orders of magnitude.

Reading (f) uses every significant connection; reading (s) the connections kept in the 500
sub-networks, each counted once per recording.

In [ ]:
def lognorm_pmf(x, a, mu, s):
    # paper, Eq 21: alpha / (sigma sqrt(2 pi)) exp(-(log10 x - mu)^2 / (2 sigma^2))
    return a / (s * np.sqrt(2 * np.pi)) * np.exp(-(np.log10(x) - mu) ** 2 / (2 * s ** 2))

def fit_r2(x, y):
    lx = np.log10(x)
    m = (y * lx).sum() / y.sum()
    sd = np.sqrt((y * (lx - m) ** 2).sum() / y.sum())
    try:
        # start from the moments of the binned distribution; alpha starts at the log bin width
        p, _ = curve_fit(lognorm_pmf, x, y, p0=[lx[1] - lx[0], m, sd], maxfev=20000)
        pred = lognorm_pmf(x, *p)
        return 1 - ((y - pred) ** 2).sum() / ((y - y.mean()) ** 2).sum(), p
    except RuntimeError:
        return np.nan, None

rows, curves = [], {}
for scale in hdn.TIME_SCALES:
    for reading in ["f", "s"]:
        for measure in ["te", "nte"]:
            vals = []
            for r in recs:
                net = nets[scale, r["name"]]
                if reading == "f":
                    v = net[measure][net["sig"]]
                else:
                    e = set()
                    for g, s_e, r_e in res[scale, r["name"]]["subs"]:
                        e.update(zip(s_e.tolist(), r_e.tolist()))
                    e = np.array(sorted(e))
                    v = net[measure][e[:, 0], e[:, 1]]
                vals.append(v[v > 0])
            pooled = np.concatenate(vals)
            edges = np.logspace(np.log10(pooled.min()), np.log10(pooled.max()), 101)
            centers = np.sqrt(edges[:-1] * edges[1:])
            pmfs = np.array([np.histogram(v, edges)[0] / len(v) for v in vals])
            r2_avg, p = fit_r2(centers, pmfs.mean(0))
            r2_each = np.array([fit_r2(centers, y)[0] for y in pmfs])
            p1, p99 = np.percentile(pooled, [1, 99])
            spread_each = np.array([np.percentile(v, 99) / np.percentile(v, 1) for v in vals])
            curves[scale, reading, measure] = (centers, pmfs, p)
            rows.append({"scale": scale, "reading": reading, "measure": measure, "n_values": len(pooled),
                         "R2_average": r2_avg, "R2_rec_median": np.nanmedian(r2_each), "R2_rec_min": np.nanmin(r2_each),
                         "p1": p1, "p99": p99, "p99/p1": p99 / p1, "p99/p1_rec_min": spread_each.min(),
                         "min": pooled.min(), "max": pooled.max(), "mu_log10": p[1] if p is not None else np.nan,
                         "sigma_log10": p[2] if p is not None else np.nan})
fig4 = pd.DataFrame(rows)
print(fig4.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print("\nCriterion (each reading): R2 >= 0.9 and p99/p1 >= 100 for both scales and both measures")
for reading in ["f", "s"]:
    sub = fig4[fig4.reading == reading]
    print(f"  reading ({reading}):", bool(((sub.R2_average >= 0.9) & (sub["p99/p1"] >= 100)).all()))

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for ax, key in zip(axes.flat, curves):
    centers, pmfs, p = curves[key]
    ax.semilogx(centers, pmfs.mean(0), "k.", ms=3)
    if p is not None:
        ax.semilogx(centers, lognorm_pmf(centers, *p), "r-")
    ax.set_title(" ".join(key), fontsize=9)
plt.tight_layout(); plt.show()
print("Every bin of the 100 log-spaced bins is shown; values span the full range of each panel's x axis.")

**Result: partially reproduced.** The significant TE values span several orders of magnitude: the 99th percentile is 8.0e4 to 2.93e5 times the 1st for raw TE and 1.84e3 to 5.43e3 times for normalized TE, at both time scales and under both readings (the smallest single-recording ratio is 2.71, normalized TE, reading s). Fits of the log-normal of Eq 21 to single recordings have a median R2 of 0.86 to 0.96, but the fit to the distribution averaged over recordings has R2 of 0.08 to 0.61, below the criterion's 0.9 in all eight cases, because the recordings' distributions sit at different places on the log axis and their average is broad and uneven. The criterion (R2 at least 0.9 on the average and a 100-fold spread) is met for the spread only.

## fig4bd-heavy-tailed-degree

> The total degree distributions are heavy-tailed, with many more high-degree neurons than in random
> networks with the same numbers of neurons and connections.

Random networks place the same number of directed connections uniformly among the same neurons:
under (f) 100 random networks per recording, under (s) one per sub-network (50 neurons, 75
connections). The threshold is the 99.9th percentile of the random total degrees of the recording
(f) or of all its random sub-networks (s).

In [ ]:
rng = np.random.default_rng(41)

def random_degrees(n, e, rng):
    flat = rng.choice(n * (n - 1), e, replace=False)
    s = flat // (n - 1); r = flat % (n - 1); r = r + (r >= s)
    return np.bincount(s, minlength=n), np.bincount(r, minlength=n)

deg = {}
rows = []
for scale in hdn.TIME_SCALES:
    for reading in ["f", "s"]:
        real_in, real_out, rand_tot, above, n_real, max_real, max_rand = [], [], [], 0, 0, 0, 0
        per_rec = []
        for r in recs:
            net = nets[scale, r["name"]]
            if reading == "f":
                sig = net["sig"]; n = sig.shape[0]; e = int(sig.sum())
                ins, outs = [sig.sum(0)], [sig.sum(1)]
                rt = np.concatenate([np.add(*random_degrees(n, e, rng)) for _ in range(100)])
            else:
                ins, outs, rt = [], [], []
                for g, s_e, r_e in res[scale, r["name"]]["subs"]:
                    loc = {v: k for k, v in enumerate(g.tolist())}
                    s_l = np.array([loc[v] for v in s_e.tolist()]); r_l = np.array([loc[v] for v in r_e.tolist()])
                    ins.append(np.bincount(r_l, minlength=len(g))); outs.append(np.bincount(s_l, minlength=len(g)))
                    rt.append(np.add(*random_degrees(len(g), len(s_e), rng)))
                rt = np.concatenate(rt)
            ins, outs = np.concatenate(ins), np.concatenate(outs)
            tot = ins + outs
            thr = np.percentile(rt, 99.9)
            above += int((tot > thr).sum()); n_real += len(tot)
            max_real = max(max_real, tot.max()); max_rand = max(max_rand, rt.max())
            per_rec.append({"rec": r["name"], "in": ins, "out": outs, "tot": tot, "rand": rt, "thr": thr})
        deg[scale, reading] = per_rec
        rows.append({"scale": scale, "reading": reading, "neurons_or_slots": n_real,
                     "fraction_above_random_99.9": above / n_real, "ratio_to_0.001": above / n_real / 0.001,
                     "max_real_total": int(max_real), "max_random_total": int(max_rand),
                     "median_threshold": np.median([p["thr"] for p in per_rec])})
fig4bd = pd.DataFrame(rows)
print(fig4bd.round(4).to_string(index=False))
print("\nCriterion: fraction >= 0.01 and max real > max random, at both scales")
for reading in ["f", "s"]:
    sub = fig4bd[fig4bd.reading == reading]
    print(f"  reading ({reading}):", bool(((sub["fraction_above_random_99.9"] >= 0.01) & (sub.max_real_total > sub.max_random_total)).all()))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, key in zip(axes, deg):
    tot = np.concatenate([p["tot"] for p in deg[key]]); rt = np.concatenate([p["rand"] for p in deg[key]])
    bins = np.arange(0, max(tot.max(), rt.max()) + 2)
    ax.loglog(bins[:-1], np.histogram(tot, bins, density=True)[0], "k.", label="real total")
    ax.loglog(bins[:-1], np.histogram(rt, bins, density=True)[0], "r.", label="random total")
    ax.set_title(" ".join(key)); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()
print("Zero-degree neurons and empty bins are not shown on the log axes.")

**Result: reproduced.** Of the neurons, 20.9% (1.6 to 6.4 ms) and 21.1% (3.5 to 14 ms) have a total degree above the 99.9th percentile of matched random networks in the full networks, 209 and 210 times the 0.001 expected; in the sub-networks 5.9% and 6.4%. The largest real total degree is 886 and 929 (full) and 53 and 54 (sub-networks), against 223, 162, 13 and 14 in the random networks.

## fig4bd-out-tail-longer

> The in-degree distribution has a shorter tail than the out-degree distribution, so there are more
> high out-degree neurons than high in-degree neurons.

In [ ]:
rows = []
for (scale, reading), per_rec in deg.items():
    ins = np.concatenate([p["in"] for p in per_rec]); outs = np.concatenate([p["out"] for p in per_rec])
    q_in = np.array([np.percentile(p["in"], 99) for p in per_rec]); q_out = np.array([np.percentile(p["out"], 99) for p in per_rec])
    w = stats.wilcoxon(q_out, q_in, alternative="greater") if np.any(q_out != q_in) else None
    rows.append({"scale": scale, "reading": reading, "p99_out": np.percentile(outs, 99), "p99_in": np.percentile(ins, 99),
                 "max_out": outs.max(), "max_in": ins.max(), "recordings_out_gt_in": int((q_out > q_in).sum()),
                 "recordings_out_lt_in": int((q_out < q_in).sum()), "wilcoxon_p_one_sided": w.pvalue if w else np.nan})
tail = pd.DataFrame(rows)
print(tail.round(4).to_string(index=False))
print("\nCriterion: pooled p99 and max of out-degree above those of in-degree, at both scales")
for reading in ["f", "s"]:
    sub = tail[tail.reading == reading]
    print(f"  reading ({reading}):", bool(((sub.p99_out > sub.p99_in) & (sub.max_out > sub.max_in)).all()))

**Result: reproduced, under the assumption that Fig 4 uses the sub-networks (reading s).** In the sub-networks the 99th percentile of out-degree is 19 and 23 against 8 and 7 for in-degree, the largest out-degree is 46 and 48 against 45 and 46, and the per-recording 99th percentile of out-degree is higher in 21 and 25 of 25 recordings (one-sided Wilcoxon p = 0.0001 and below 0.0001). In the full networks (reading f) the 99th percentiles also favour out-degree (242 against 234, 228.7 against 179; Wilcoxon p = 0.0015 and below 0.0001), but at 3.5 to 14 ms the largest in-degree (480) exceeds the largest out-degree (449), so that reading fails the criterion's second part.

## fig5a-in-degree-uncorrelated, fig5b-out-degree-positive

> fig5a: The correlation between a receiver's in-degree and the synergy (computation) it performs is
> centered near zero across recordings, with a slight negative skew; computation is independent of
> in-degree.

> fig5b: Most recordings show a positive correlation between a transmitter's out-degree and the
> synergy computed by its receivers.

Synergy is normalized by H(I_F). Under (s) each triple of each sub-network is a data point, with
degrees counted inside the sub-network, the 500 sub-networks pooled per recording; under (f) the
triples of the full network (sampled, see the table above) with full-network degrees. Each
transmitter of a triple is a separate data point for the out-degree. Correlations: (p) Pearson,
(r) Spearman.

In [ ]:
def triple_data(scale, name, reading):
    a = res[scale, name]
    if reading == "s":
        t = a["sub_triples"]
        syn = np.array([a["syn"][(x[1], x[2], x[3])] for x in t])
        return t[:, 4], syn, np.r_[t[:, 5], t[:, 6]], np.r_[syn, syn]
    t = a["full_triples"]
    syn = np.array([a["syn"][tuple(x)] for x in t.tolist()])
    return a["indeg_f"][t[:, 0]], syn, np.r_[a["outdeg_f"][t[:, 1]], a["outdeg_f"][t[:, 2]]], np.r_[syn, syn]

corr = []
for scale in hdn.TIME_SCALES:
    for r in recs:
        for reading in ["s", "f"]:
            indeg, syn_in, outdeg, syn_out = triple_data(scale, r["name"], reading)
            for kind, f in [("p", stats.pearsonr), ("r", stats.spearmanr)]:
                for panel, x, y in [("in", indeg, syn_in), ("out", outdeg, syn_out)]:
                    ok = len(x) > 2 and np.ptp(x) > 0
                    c = f(x, y) if ok else None
                    corr.append({"scale": scale, "recording": r["name"], "reading": reading, "coef": kind, "panel": panel,
                                 "n": len(x), "r": c[0] if ok else np.nan, "p": c[1] if ok else np.nan})
corr = pd.DataFrame(corr)
print("Correlations that could not be computed (fewer than 3 points or one degree):", int(corr.r.isna().sum()))

def skew_test(r):
    r = r.dropna(); pos, neg = int((r > 0).sum()), int((r < 0).sum()); n = pos + neg
    return {"n": n, "positive": pos, "negative": neg, "median_r": r.median(),
            "P(neg skew this large or larger)": stats.binom.cdf(pos, n, 0.5),
            "P(pos skew this large or larger)": stats.binom.sf(pos - 1, n, 0.5)}

skew = []
for panel in ["in", "out"]:
    for reading in ["s", "f"]:
        for kind in ["p", "r"]:
            for scale in ["both"] + list(hdn.TIME_SCALES):
                sub = corr[(corr.panel == panel) & (corr.reading == reading) & (corr.coef == kind)]
                if scale != "both":
                    sub = sub[sub.scale == scale]
                skew.append({"panel": panel, "reading": reading, "coef": kind, "scale": scale, **skew_test(sub.r)})
skew = pd.DataFrame(skew)
print(skew.to_string(index=False, float_format=lambda v: f"{v:.3g}"))

In [ ]:
print("Per-recording correlations, reading (s), Pearson, by time scale:")
piv = corr[(corr.reading == "s") & (corr.coef == "p")].pivot_table(index="recording", columns=["panel", "scale"], values="r")
print(piv.round(3).to_string())
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, panel in zip(axes, ["in", "out"]):
    v = corr[(corr.reading == "s") & (corr.coef == "p") & (corr.panel == panel)].r.dropna()
    ax.hist(v, bins=np.linspace(-1, 1, 41)); ax.axvline(0, color="k")
    ax.set_title(f"{panel}-degree vs synergy, (s), Pearson; range {v.min():.2f} to {v.max():.2f}")
plt.tight_layout(); plt.show()

**Result: partially reproduced, under the assumption of reading (s) with Pearson correlations.** Correlations between receiver in-degree and synergy are mostly negative: 41 of 50 under (s, Pearson), median r = -0.14, binomial probability of a negative skew this large 2.8e-6. That is the paper's slight negative skew, but significant, which the criterion counts as partial. Three of the four readings give a significant negative skew (40 or 41 of 50 negative); the full network with Spearman correlations gives 30 of 50 negative (p = 0.10, not significant), which would count as reproduced.

**Result: reproduced, under the assumption of reading (s).** Correlations between transmitter out-degree and synergy are positive in 44 of 50 recordings and time scales under (s, Pearson), median r = 0.19, binomial p = 1.6e-8; 39 of 50 under (s, Spearman), p = 4.5e-5; 35 of 50 under (f, Pearson), p = 0.0033. Under (f, Spearman) 26 of 50 are positive (p = 0.44), a majority without significance, which the criterion counts as partial.

## fig5ab-correlations-beyond-null

> Nearly all the per-recording degree and synergy correlations lie beyond a null of randomized degree
> and computation pairings, whose correlations sit near zero.

Ten permutations of degree against synergy per correlation (400 null values for 40 correlations
in the paper; here 10 for each of the 50).

In [ ]:
rng = np.random.default_rng(52)
rows = []
for reading in ["s", "f"]:
    for kind, f in [("p", stats.pearsonr), ("r", stats.spearmanr)]:
        for panel in ["in", "out"]:
            real, null = [], []
            for scale in hdn.TIME_SCALES:
                for r in recs:
                    indeg, syn_in, outdeg, syn_out = triple_data(scale, r["name"], reading)
                    x, y = (indeg, syn_in) if panel == "in" else (outdeg, syn_out)
                    if len(x) < 3 or np.ptp(x) == 0:
                        continue
                    real.append(f(x, y)[0])
                    null += [f(rng.permutation(x), y)[0] for _ in range(10)]
            real, null = np.abs(np.array(real)), np.abs(np.array(null))
            thr = np.percentile(null, 99)
            rows.append({"reading": reading, "coef": kind, "panel": panel, "n_corr": len(real), "n_null": len(null),
                         "null_|r|_median": np.median(null), "null_|r|_p99": thr,
                         "fraction_beyond": (real > thr).mean()})
beyond = pd.DataFrame(rows)
print(beyond.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print("\nCriterion: at least 90% beyond the null's 99th percentile in each panel")
for (reading, kind), sub in beyond.groupby(["reading", "coef"]):
    print(f"  ({reading}, {kind}):", bool((sub.fraction_beyond >= 0.9).all()))

**Result: reproduced, under the assumption of reading (s).** Under (s) the null correlations from permuted degrees have a median absolute value of 0.002 to 0.003 and a 99th percentile of 0.008 to 0.013; 100% and 96% (Pearson) and 94% and 96% (Spearman) of the in-degree and out-degree correlations lie beyond it. Under (f) the fractions are 84% and 92% (Pearson), below 90% for in-degree, and 90% and 98% (Spearman).

## fig5cd-pooled-trends

> Pooled over all neuron groups and recordings, synergy shows no trend with receiver in-degree and
> rises with transmitter out-degree, at both time scales.

In [ ]:
rows = []
for reading in ["s", "f"]:
    for scale in hdn.TIME_SCALES:
        parts = [triple_data(scale, r["name"], reading) for r in recs]
        for panel in ["in", "out"]:
            x = np.concatenate([p[0] if panel == "in" else p[2] for p in parts])
            y = np.concatenate([p[1] if panel == "in" else p[3] for p in parts])
            d = pd.DataFrame({"deg": x, "syn": y}).groupby("deg").syn.agg(["count", "median"])
            d = d[d["count"] >= 20]
            rho, p = stats.spearmanr(d.index, d["median"]) if len(d) > 2 else (np.nan, np.nan)
            rows.append({"reading": reading, "scale": scale, "panel": panel, "degrees_kept": len(d),
                         "degree_range": f"{d.index.min()} to {d.index.max()}", "groups": int(d["count"].sum()),
                         "median_at_lowest": d["median"].iloc[0], "median_at_highest": d["median"].iloc[-1],
                         "spearman_rho": rho, "p": p})
trend = pd.DataFrame(rows)
print(trend.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print("\nCriterion: out-degree rho > 0 with p < 0.05, and in-degree not significantly positive, at both scales")
for reading in ["s", "f"]:
    sub = trend[trend.reading == reading]
    o, i = sub[sub.panel == "out"], sub[sub.panel == "in"]
    print(f"  reading ({reading}):", bool(((o.spearman_rho > 0) & (o.p < 0.05)).all() and not ((i.spearman_rho > 0) & (i.p < 0.05)).any()))

**Result: not reproduced.** With all triples pooled, the per-degree median synergy falls steeply with receiver in-degree (Spearman rho = -0.96 and -0.94 under (s), -0.51 and -0.81 under (f)), which is not significantly positive, as the criterion requires, but is a trend where the paper reports none. The rise with transmitter out-degree is significant at 1.6 to 6.4 ms under (s) (rho = 0.57, p = 4e-5) and at 3.5 to 14 ms under (f) (rho = 0.34, p = 1.2e-8), but not at the other scale of either reading (rho = 0.14, p = 0.34 under (s); rho = -0.51 under (f)). No reading meets the criterion at both time scales.

## fig6-information-gain-not-rising

> The information gained by adding inputs to a neuron, a bound on its highest-order synergy, stays
> constant or decreases with the number of inputs at both time scales; exponential fits to the
> medians give small negative exponents, one with a 95% range overlapping zero.

Inputs are the significant inputs of the full network (reading f); reading (s) was not run (the
count of sub-network receivers with six inputs is printed below).
Per recording and time scale, 30 receivers with at least one input are sampled, and for each n up to
20 combinations of n inputs (the paper: every receiver, up to 100 combinations).

In [ ]:
def expo(n, A, k):
    return A * np.exp(k * n)

n6 = {scale: int(sum((np.bincount(r_e) >= 6).sum() for (sc, nm), a in res.items() if sc == scale
                     for g, s_e, r_e in a["subs"])) for scale in hdn.TIME_SCALES}
print("Receivers with 6 or more inputs inside a sub-network, summed over sub-networks:", n6)
rows = []
for scale in hdn.TIME_SCALES:
    for norm, col in [("u (normalized)", 1), ("v (bits)", 0)]:
        per_n = {}
        for n_in in range(1, hdn.MAX_INPUTS + 1):
            per_n[n_in] = np.array([v[col] for (sc, nm), a in res.items() if sc == scale for v in a["gains"].get(n_in, {}).values()])
        for first in [2, 1]:
            ns = np.arange(first, hdn.MAX_INPUTS + 1)
            med = np.array([np.median(per_n[n]) for n in ns])
            p, cov = curve_fit(expo, ns, med, p0=[med[0] * np.exp(0.1 * first), -0.1], maxfev=20000)
            se = np.sqrt(cov[1, 1]); tq = stats.t.ppf(0.975, len(ns) - 2)
            rows.append({"scale": scale, "norm": norm, "n_from": first,
                         "neurons_per_n": "/".join(str(len(per_n[n])) for n in ns),
                         "medians": " ".join(f"{m:.3g}" for m in med), "k": p[1],
                         "k_low": p[1] - tq * se, "k_high": p[1] + tq * se})
gain = pd.DataFrame(rows)
print(gain.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
for (norm, first), sub in gain.groupby(["norm", "n_from"]):
    allneg = bool((sub.k < 0).all()); none_above = bool((sub.k_low <= 0).all())
    print(f"  ({norm}, n from {first}): k < 0 at both scales: {allneg}; no interval wholly above zero: {none_above}")

**Result: reproduced (on a sample of receivers).** The median information gain falls with the number of inputs at both time scales; fitted exponents are negative under all four readings (normalized or in bits, n from 1 or from 2), from -0.037 to -0.115, and no 95% interval lies above zero. The statement's further detail, that one of the two intervals overlaps zero, does not hold here: every interval lies below zero (for normalized gain from n = 2, -0.120 to -0.036 and -0.098 to -0.057). The gains come from 30 sampled receivers per recording and time scale (626 to 750 neurons per n) and at most 20 combinations per neuron and n.

## table1-pid-examples

> On the example distributions of Table 1, the PID multivariate TE finds synergy only in the
> synergistic and the synergistic and redundant examples; mutual information misses the hidden self
> interaction, and interaction information misses simultaneous synergy and redundancy and unique
> information.

The joint distributions are the paper's Table 1 (rows I_F, I_P, J_P, K_P from 0000 to 1111), typed
in as printed; each column is renormalized to sum to 1.

In [ ]:
TABLE1 = {  # p(I_F, I_P, J_P, K_P), rows 0000 ... 1111, as printed in Table 1
    "No Interaction": ["1/16"] * 16,
    "Self Interaction": ["1/8", "1/8", "1/8", "1/8", "0", "0", "0", "0", "0", "0", "0", "0", "1/8", "1/8", "1/8", "1/8"],
    "Hidden Self Interaction": ["1/4", "1/4", "0", "0", "0", "0", "0", "0", "0", "0", "0", "0", "0", "0", "1/4", "1/4"],
    "Single Interaction": ["1/8", "1/8", "0", "0", "1/8", "1/8", "0", "0", "0", "0", "1/8", "1/8", "0", "0", "1/8", "1/8"],
    "Redundant Interaction": ["1/4", "0", "0", "0", "1/4", "0", "0", "0", "0", "0", "0", "1/4", "0", "0", "0", "1/4"],
    "Single and Redundant Interaction": ["0.1964", "0.0536", "0", "0", "0.1964", "0.0536", "0", "0", "0", "0", "0.0536", "0.1964", "0", "0", "0.0536", "0.1964"],
    "Synergistic Interaction": ["1/8", "0", "0", "1/8", "1/8", "0", "0", "1/8", "0", "1/8", "1/8", "0", "0", "1/8", "1/8", "0"],
    "Synergistic and Redundant Interaction": ["0.0548", "0.0548", "0.0548", "0", "0.0548", "0.0548", "0.0548", "0", "0", "0", "0", "0.3357", "0", "0", "0", "0.3357"],
}

def mi(pxy):
    px, py = pxy.sum(1, keepdims=True), pxy.sum(0, keepdims=True)
    m = pxy > 0
    return float((pxy[m] * np.log2(pxy[m] / (px @ py)[m])).sum())

rows = []
for name, col in TABLE1.items():
    p = np.array([eval(v) for v in col], float); p = p / p.sum()
    n = np.zeros((2, 2, 4))
    pf = np.zeros((2, 2, 2))   # p(I_F, J_P, K_P)
    for row in range(16):
        f, q, j, k = (row >> 3) & 1, (row >> 2) & 1, (row >> 1) & 1, row & 1
        n[f, q, j + 2 * k] = p[row] * 1e6
        pf[f, j, k] += p[row]
    syn, red, uJ, uK, teJ, teK, teJK, H = hdn.pid_from_table(n)
    mi_j = mi(pf.sum(2))                                  # MI(J_P; I_F)
    mi_jk = mi(pf.sum(0))                                 # MI(J_P; K_P)
    mi_jk_f = sum(pf[f].sum() * mi(pf[f] / pf[f].sum()) for f in range(2) if pf[f].sum() > 0)  # MI(J_P; K_P | I_F)
    rows.append({"example": name, "TE(J)": teJ, "TE(K)": teK, "TE(JK)": teJK, "redundancy": red, "unique J": uJ,
                 "unique K": uK, "synergy": syn, "MI(J_P;I_F)": mi_j, "II": mi_jk_f - mi_jk})
t1 = pd.DataFrame(rows).set_index("example")
print(t1.round(4).to_string())
tol = 0.001
hs, srd, snr = t1.loc["Hidden Self Interaction"], t1.loc["Synergistic and Redundant Interaction"], t1.loc["Single and Redundant Interaction"]
checks = {
    "synergy > 0.001 exactly in the two synergistic examples":
        set(t1.index[t1.synergy > tol]) == {"Synergistic Interaction", "Synergistic and Redundant Interaction"},
    "hidden self: MI(J_P;I_F) > 0 while TE and every PID term are 0":
        hs["MI(J_P;I_F)"] > tol and all(abs(hs[c]) <= tol for c in ["TE(J)", "TE(K)", "TE(JK)", "redundancy", "unique J", "unique K", "synergy"]),
    "synergistic and redundant: II = 0 while synergy and redundancy are not":
        abs(srd["II"]) <= tol and srd.synergy > tol and srd.redundancy > tol,
    "single and redundant: unique J > 0 and |II| = redundancy":
        snr["unique J"] > tol and abs(abs(snr["II"]) - snr.redundancy) <= tol,
}
for k, v in checks.items():
    print(f"{k}: {v}")

**Result: reproduced.** On the distributions of Table 1, synergy is above 0.001 bits only in the synergistic (1.000 bit) and the synergistic and redundant (0.457 bits) examples. In the hidden self interaction MI(J_P; I_F) is 1 bit while TE and every PID term are 0. In the synergistic and redundant example the interaction information is 0.0001 bits while synergy and redundancy are 0.457 bits each. In the single and redundant example the unique information of J is 0.750 bits and the interaction information is -0.250 bits, the size of the redundancy (0.250 bits).

## Feedforward model: fig7b, fig7c, fig7d, figs1

> fig7c: In a two-layer feedforward model, degree-modified and firing-rate-modified Hebbian rewiring
> rules qualitatively reproduce the degree and synergy correlations of the real data, the
> degree-modified rule more closely, while a pure Hebbian rule and random wiring do not.

> fig7b: The pure Hebbian rule pools connections among the input neurons most correlated with the
> network-wide signal, the firing-rate-modified rule spreads them to less correlated input neurons,
> and the degree-modified rule spreads them almost uniformly across the input layer.

> fig7d: The pure Hebbian rule gives the highest mutual information between connected neurons, but
> conditioned on the network-wide binary signal the degree-modified rule gives the highest; it alone
> raises neuron-to-neuron communication while also reinforcing network-wide correlation.

> figs1: With 40 neurons per layer the model gives the same qualitative results, the degree-modified
> rule again raising neuron-to-neuron communication, though the Hebbian and firing-rate-modified
> rules give larger conditioned mutual information than in the small model.

The model follows the Methods and Eqs 13 to 24 of the version of record (`hdn.py`). Synergy and MI
are computed from 2000 fresh runs per model. The real data's reference is the mean and standard
deviation of the 50 per-recording correlations (25 recordings, two time scales) of reading (s),
Pearson, from the fig5 section; the other readings are shown.

In [ ]:
MODEL_EVAL_RUNS = 2000
models = {}
t_all = time.time()
for N, readings in [(20, ["kept"]), (40, ["recomputed", "kept"])]:
    for ab in readings:
        alpha, beta = hdn.sigmoid_params(N if ab == "recomputed" else 20)
        for k, rule in enumerate(hdn.RULES[N]):
            rng = np.random.default_rng(700 + 10 * N + k + (5 if ab == "kept" and N == 40 else 0))
            conn = hdn.build_models(N, rule, rng, alpha, beta)
            models[N, ab, rule] = (conn, pd.DataFrame(hdn.model_correlations(conn, rng, alpha, beta, MODEL_EVAL_RUNS)))
print(f"models built and evaluated in {time.time() - t_all:.0f} s")
print("alpha, beta for N = 20:", [round(v, 4) for v in hdn.sigmoid_params(20)], "| recomputed for N = 40:", [round(v, 4) for v in hdn.sigmoid_params(40)])
rows = []
for (N, ab, rule), (conn, d) in models.items():
    rows.append({"N": N, "alpha_beta": ab, "rule": rule, "models": len(d), "connections": int(conn[0].sum()),
                 **{c: d[c].mean() for c in ["in_p", "out_p", "in_r", "out_r", "mi", "mi_given_b", "upper_half_fraction"]},
                 "in_p_sd": d.in_p.std(), "out_p_sd": d.out_p.std(), "nan_in": int(d.in_p.isna().sum()), "nan_out": int(d.out_p.isna().sum())})
mod = pd.DataFrame(rows)
print(mod.to_string(index=False, float_format=lambda v: f"{v:.4g}"))

In [ ]:
def real_ref(reading, kind):
    sub = corr[(corr.reading == reading) & (corr.coef == kind)]
    i, o = sub[sub.panel == "in"].r.dropna(), sub[sub.panel == "out"].r.dropna()
    return i.mean(), i.std(), o.mean(), o.std()

def match_table(N, ab, reading, kind):
    mi_, si, mo, so = real_ref(reading, kind)
    rows = []
    for rule in hdn.RULES[N]:
        d = models[N, ab, rule][1]
        a, b = d["in_" + kind].mean(), d["out_" + kind].mean()
        rows.append({"rule": rule, "mean_in": a, "mean_out": b, "match": abs(a - mi_) <= si and abs(b - mo) <= so,
                     "distance": math.hypot(a - mi_, b - mo)})
    return pd.DataFrame(rows), (mi_, si, mo, so)

def fig7c_ok(t):
    m = t.set_index("rule")
    return bool(m.match["degree"] and m.match["firing_rate"] and not m.match["hebbian"] and not m.match["random"]
                and m.distance["degree"] < m.distance["firing_rate"])

for reading in ["s", "f"]:
    for kind in ["p", "r"]:
        t, ref = match_table(20, "kept", reading, kind)
        print(f"reading ({reading}, {kind}): real in {ref[0]:.3f} +- {ref[1]:.3f}, out {ref[2]:.3f} +- {ref[3]:.3f}")
        print(t.round(4).to_string(index=False))
        print("  fig7c criterion:", fig7c_ok(t))
fig, ax = plt.subplots(figsize=(5, 4))
for rule in hdn.RULES[20]:
    d = models[20, "kept", rule][1]
    ax.plot(d.in_p, d.out_p, ".", alpha=0.3, label=rule)
mi_, si, mo, so = real_ref("s", "p")
ax.errorbar(mi_, mo, xerr=si, yerr=so, color="k", label="real (s, p)")
ax.set_xlabel("in-degree vs synergy r"); ax.set_ylabel("out-degree vs synergy r"); ax.legend(fontsize=7)
plt.show()
d_all = pd.concat([models[20, "kept", r][1] for r in hdn.RULES[20]])
print(f"model correlations range: in {d_all.in_p.min():.3f} to {d_all.in_p.max():.3f}, out {d_all.out_p.min():.3f} to {d_all.out_p.max():.3f}; all within the axes")

**Result: partially reproduced, under the assumption of reading (s) with Pearson correlations for the real data.** Against the real mean and standard deviation (in-degree -0.132 +- 0.155, out-degree 0.195 +- 0.170), the degree-modified (0.017, 0.114) and firing-rate-modified (-0.061, 0.280) models match, and the Hebbian (0.321, 0.022) and random (0.128, -0.009) models do not, as the paper says. But the firing-rate-modified rule is closer to the real data (distance 0.111) than the degree-modified rule (0.169), so the criterion's last part fails; it fails too under (s, Spearman) and (f, Pearson), and only (f, Spearman) meets the whole criterion. In 56 of 100 degree-modified models every output neuron ends with the same in-degree, so the in-degree correlation is undefined there; its mean is over the other 44.

In [ ]:
def fig7b_table(N, ab):
    t = pd.DataFrame([{"rule": r, "upper_half_fraction": models[N, ab, r][1].upper_half_fraction.mean(),
                       "sd": models[N, ab, r][1].upper_half_fraction.std()} for r in hdn.RULES[N]])
    m = t.set_index("rule").upper_half_fraction
    ok = bool(m["hebbian"] > m["firing_rate"] > m["degree"] and abs(m["degree"] - 0.5) <= 0.1)
    return t, ok

t, ok = fig7b_table(20, "kept")
print(t.round(4).to_string(index=False)); print("fig7b criterion:", ok)
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, rule in zip(axes, hdn.RULES[20]):
    im = ax.imshow(models[20, "kept", rule][0].mean(0), vmin=0, vmax=1, cmap="viridis")
    ax.set_title(rule); ax.set_xlabel("output o"); ax.set_ylabel("input i")
plt.colorbar(im, ax=axes); plt.show()

**Result: reproduced.** The fraction of connections leaving the upper half of the input layer is 0.988 for the Hebbian rule, 0.792 for the firing-rate-modified rule and 0.505 for the degree-modified rule (random wiring 0.502), in the order the criterion requires, with the degree-modified rule within 0.5 +- 0.1.

In [ ]:
def fig7d_tests(N, ab):
    rules = list(hdn.RULES[N])
    rows = []
    for col in ["mi", "mi_given_b"]:
        for a, b in itertools.combinations(rules, 2):
            x, y = models[N, ab, a][1][col], models[N, ab, b][1][col]
            rows.append({"measure": col, "a": a, "b": b, "mean_a": x.mean(), "mean_b": y.mean(),
                         "p": stats.mannwhitneyu(x, y, alternative="two-sided").pvalue})
    t = pd.DataFrame(rows)
    t["p_fdr"] = stats.false_discovery_control(t.p, method="bh")
    return t

def fig7d_ok(N, ab, t):
    means = {r: models[N, ab, r][1][["mi", "mi_given_b"]].mean() for r in hdn.RULES[N]}
    top_u = max(means, key=lambda r: means[r]["mi"]); top_c = max(means, key=lambda r: means[r]["mi_given_b"])
    c = t[t.measure == "mi_given_b"]
    deg_beats = all(((c.a == "degree") & (c.b == r) | (c.b == "degree") & (c.a == r)).any() and
                    c[((c.a == "degree") & (c.b == r)) | ((c.b == "degree") & (c.a == r))].p_fdr.iloc[0] < 0.001
                    for r in hdn.RULES[N] if r != "degree")
    not_above_random = all(not (means[r]["mi_given_b"] > means["random"]["mi_given_b"] and
                                c[((c.a == r) & (c.b == "random")) | ((c.b == r) & (c.a == "random"))].p_fdr.iloc[0] < 0.001)
                           for r in ["hebbian", "firing_rate"])
    return top_u, top_c, deg_beats, not_above_random

t = fig7d_tests(20, "kept")
print(t.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
top_u, top_c, deg_beats, nar = fig7d_ok(20, "kept", t)
print(f"highest unconditioned MI: {top_u}; highest conditioned MI: {top_c}; degree above every other rule at FDR p < 0.001: {deg_beats}; "
      f"Hebbian and firing-rate not significantly above random: {nar}")
print("fig7d criterion:", top_u == "hebbian" and top_c == "degree" and deg_beats and nar)

**Result: not reproduced.** The Hebbian rule gives the highest unconditioned MI between connected neurons (0.090 bits, against 0.071, 0.030 and 0.023), as the paper says. Conditioned on the binary signal, the random model gives the highest MI (0.00230 bits) and the degree-modified rule the lowest (0.00153; Hebbian 0.00168, firing-rate 0.00166); the degree-modified rule differs from each other rule at FDR p < 0.001 but is below them, not above. All conditioned values are small (0.0015 to 0.0023 bits), so the ordering rests on small differences.

In [ ]:
for ab in ["recomputed", "kept"]:
    print(f"=== N = 40, alpha and beta {ab} ===")
    ok_all = []
    for reading, kind in [("s", "p"), ("s", "r"), ("f", "p"), ("f", "r")]:
        t, ref = match_table(40, ab, reading, kind)
        m = t.set_index("rule")
        ok_c = bool(m.match["degree"] and m.match["firing_rate"] and not m.match["hebbian"])
        print(f"reading ({reading}, {kind}):", t.round(4).to_dict("list"), "| correlation match:", ok_c)
        ok_all.append(ok_c)
    tb, ok_b = fig7b_table(40, ab)
    print(tb.round(4).to_string(index=False), "| connectivity ordering:", ok_b)
    td = fig7d_tests(40, ab)
    c = td[(td.measure == "mi_given_b") & (((td.a == "degree") & (td.b == "random")) | ((td.b == "degree") & (td.a == "random")))]
    deg_above = bool(models[40, ab, "degree"][1].mi_given_b.mean() > models[40, ab, "random"][1].mi_given_b.mean() and c.p_fdr.iloc[0] < 0.001)
    larger = {r: (models[40, ab, r][1].mi_given_b.mean(), models[20, "kept", r][1].mi_given_b.mean()) for r in ["hebbian", "firing_rate"]}
    print(td.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
    print("degree conditioned MI above random (FDR p < 0.001):", deg_above)
    print("conditioned MI, N = 40 against N = 20:", {r: (round(a, 5), round(b, 5)) for r, (a, b) in larger.items()})
    print("figs1 criterion, reading (s, p):", ok_all[0] and ok_b and deg_above and all(a > b for a, b in larger.values()))

**Result: not reproduced.** With 40 neurons per layer the connectivity ordering holds (Hebbian 0.991, firing-rate 0.769, degree 0.502 with alpha and beta recomputed; 0.999, 0.798, 0.501 with them kept). Under (s, Pearson) the degree-modified rule does not match the real correlations with recomputed alpha and beta (it does with them kept, where the firing-rate rule does not), its conditioned MI is below the random model's under both readings (0.00094 against 0.00112; 0.00100 against 0.00206), and the Hebbian and firing-rate rules' conditioned MI is smaller than in the 20-neuron model (0.00096 and 0.00097, or 0.00147 and 0.00146, against 0.00168 and 0.00166), not larger.

## Summary

| Claim | Outcome |
|---|---|
| data-recordings | reproduced |
| fig4ac-te-lognormal | partially reproduced |
| fig4bd-heavy-tailed-degree | reproduced |
| fig4bd-out-tail-longer | reproduced (assumes sub-networks) |
| fig5a-in-degree-uncorrelated | partially reproduced (assumes sub-networks, Pearson) |
| fig5b-out-degree-positive | reproduced (assumes sub-networks) |
| fig5ab-correlations-beyond-null | reproduced (assumes sub-networks) |
| fig5cd-pooled-trends | not reproduced |
| fig6-information-gain-not-rising | reproduced |
| table1-pid-examples | reproduced |
| fig7b-connectivity-patterns | reproduced |
| fig7c-degree-modified-rule-matches | partially reproduced (assumes sub-networks, Pearson) |
| fig7d-degree-rule-highest-conditioned-mi | not reproduced |
| figs1-larger-model | not reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The paper reused TE networks computed for [44]; they are rebuilt here from the deposited spike times with the bins, past states, jitter window and threshold of [44], so they need not be the same networks. Which two bins form the past state is read from the delay windows both papers state, not from [44]'s text, which implies other bins. For pairs whose jitter moves more than 50 relevant spikes the surrogate counts are drawn from a normal distribution with their exact moments; on two recordings this agrees with the full jitter on 99.5% and 99.6% of the significant connections. The full-network triples are sampled (3000 per recording and time scale), and the information gain uses 30 receivers per recording and time scale and at most 20 combinations per n (the paper: all receivers, 100 combinations). The model's synergy is the PID of two input states about the output state in the same run, without the receiver's past, which the model does not have; MI and synergy use 2000 fresh runs per model.

What was found about the data set. The deposit matches the paper's description: 25 recordings, 7735 neurons, all with at least 100 spikes, a mean rate of 2.10 Hz, spike times on the 20 kHz grid. One recording (DataSet19) lasts 48 min, not 60. The first ten recordings have no spikes in their first 8 s. The networks of [44] that the paper analysed are not deposited, so the networks have to be rebuilt.

In [ ]:
print(f"Notebook ran in {(time.time() - started) / 60:.1f} min")